# 🎯 Multimodal Processing with Local Long-Term Memory

This notebook demonstrates multimodal content processing with persistent memory. You'll build an agent that remembers previous conversations and content across sessions using the Strands Agents [`MemoryManager`](https://strandsagents.com/docs/user-guide/sdk/memory/overview/) with the built-in [`FileMemoryStore`](https://strandsagents.com/docs/user-guide/sdk/memory/file-memory-store/), which keeps memories as local markdown files.

## What You'll Learn

- Give an agent long-term memory with `MemoryManager` + `FileMemoryStore`
- Let the agent extract durable facts automatically (`extraction=True`) and store facts on demand (`add_memory` tool)
- Maintain context across multiple sessions (new agent instances, same store)
- Process images, documents and videos with memory-enhanced responses

## Prerequisites

- AWS account with [Amazon Bedrock access](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)
- Claude Haiku 5.5 (`global.anthropic.claude-haiku-5-5`) and Amazon Nova 2 Lite (`us.amazon.nova-2-lite-v1:0`) enabled in Bedrock
- An S3 bucket for the `video_reader` tool (set `VIDEO_READER_S3_BUCKET`)

## Memory Architecture

> **Why no FAISS / Mem0 anymore?** This notebook previously used the `mem0_memory` tool from `strands-agents-tools` with a local FAISS index. That tool was removed in `strands-agents-tools` 0.9.0 ([migration guide](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/tools/community-tools-package/)). The SDK now ships memory natively: `MemoryManager` searches the store before each turn, injects relevant memories into the prompt, and (with extraction enabled) writes new facts after each turn. `FileMemoryStore` is the zero-infrastructure local option; if you prefer Mem0 you can plug in [`Mem0MemoryStore` from `mem0-strands`](https://docs.mem0.ai/integrations/strands) as another store.

For production deployments, consider [Amazon S3 Vectors](https://docs.aws.amazon.com/AmazonS3/latest/userguide/s3-vectors.html) as shown in the S3 vectors notebook, or [Amazon Bedrock Knowledge Bases](https://strandsagents.com/docs/user-guide/sdk/memory/bedrock-knowledge-base/) as a managed memory store.


In [ ]:
import os
import boto3

from strands import Agent
from strands.models import BedrockModel
from strands.memory import MemoryManager, MemoryInjectionConfig
from strands.vended_memory_stores.file_memory_store import FileMemoryStore
from strands.vended_tools import read
from video_reader import video_reader

# The tools need absolute paths
DATA = os.path.abspath("data-sample")

# video_reader uploads local videos to this S3 bucket before calling Bedrock. ⚠ CHANGE THIS!
os.environ.setdefault("VIDEO_READER_S3_BUCKET", "your-video-bucket-name")

USER_ID = "eli_abc"  # one store per user keeps memories isolated
print("✅ Imports successful!")


## 🤖 Agent Configuration with Local Memory

`FileMemoryStore(name=..., extraction=True)` writes memories as markdown under `./.strands/memory/<name>/`. `MemoryManager` registers a `search_memory` tool, injects relevant memories before each model call, and with `add_tool_config=True` also exposes an `add_memory` tool so the model can store facts explicitly.


In [ ]:
MULTIMODAL_SYSTEM_PROMPT = f""" You are a helpful assistant that can process documents, images, and videos.
Analyze their contents and provide relevant information. You have long-term memory and can remember previous interactions.

You can:

1. For PNG, JPEG/JPG, GIF, WebP, PDF, csv, docx, xls or xlsx formats use the `read` tool
2. For MP4, MOV, AVI, MKV, WebM formats use `video_reader` to process the file
3. Use `add_memory` to store facts the user asks you to remember, and `search_memory` to recall them
4. Just deliver the answer

Files live in {DATA}; always pass absolute paths to the tools.

When displaying responses:
- Format answers data in a human-readable way
- Highlight important information
- Reference relevant past interactions when appropriate
- Always reply in the original user language
"""

# Configure AWS session for Bedrock access
session = boto3.Session(region_name='us-east-1')

# Claude Haiku 5.5 (cross-region inference profile)
bedrock_model = BedrockModel(
    model_id="global.anthropic.claude-haiku-5-5",
    boto_session=session,
    streaming=False
)

# Local file-based memory store, one per user. extraction=True distills facts from each turn.
memory_store = FileMemoryStore(name=f"multimodal-{USER_ID}", extraction=True, max_search_results=10)

def create_agent():
    """New agent instance (new session) that shares the same long-term memory store."""
    return Agent(
        system_prompt=MULTIMODAL_SYSTEM_PROMPT,
        tools=[read, video_reader],
        model=bedrock_model,
        # Inject up to 10 relevant memories before each model call (default is 5)
        memory_manager=MemoryManager(
            stores=[memory_store],
            add_tool_config=True,
            injection=MemoryInjectionConfig(max_entries=10),
        ),
    )

multimodal_agent = create_agent()
print("✅ Agent with long-term memory created. Tools:", multimodal_agent.tool_names)


## 🎯 Store some facts about the user

The agent can call `add_memory` directly when asked to remember something. Extraction also runs automatically after the turn.


In [ ]:
content = """Hello, my name is Elizabeth, but they call me Eli. I'm a developer advocate at AWS, and I want to understand what's in images, videos, and documents to improve my day-to-day work. Please remember this."""
response = multimodal_agent(content)
print(response)


## 🎯 Usage Examples with Memory


In [ ]:
# Example 1: Image analysis with memory storage
print("=== 📸 IMAGE ANALYSIS WITH MEMORY ===")
image_result = multimodal_agent(f"Analyze the image {DATA}/diagram.jpg in detail and describe everything you observe. Remember the AWS services it uses for later.")
print(image_result)
print("\n" + "="*80 + "\n")


In [ ]:
# Example 2: Video analysis with memory storage
print("=== 🎬 VIDEO ANALYSIS WITH MEMORY ===")
video_result = multimodal_agent(f"Analyze the video {DATA}/moderation-video.mp4 and describe in detail the actions and scenes you observe. Store a short summary in your memory.")
print(video_result)
print("\n" + "="*80 + "\n")


In [ ]:
# Example 3: Document analysis with memory storage
print("=== 📄 DOCUMENT ANALYSIS WITH MEMORY ===")
doc_result = multimodal_agent(f"Summarize as json the content of the document {DATA}/Welcome-Strands-Agents-SDK.pdf and store the key points in your memory.")
print(doc_result)


## 🧠 Memory recall in a new session

A brand new `Agent` has an empty conversation history, but it shares the same `FileMemoryStore`, so it still knows what happened before.

> `FileMemoryStore` retrieval is keyword based (token overlap against the file name and body), so recall works best when the question shares words with the stored facts. The [file memory store docs](https://strandsagents.com/docs/user-guide/sdk/memory/file-memory-store/) show how to switch to a BM25 search strategy.


In [ ]:
new_session_agent = create_agent()
print(f"💬 Messages in the new session: {len(new_session_agent.messages)}")

print("=== 🧠 MEMORY RECALL TEST ===")
memory_result = new_session_agent("What do you remember about Eli (name, role at AWS) and about the diagram image, the moderation video, and the Strands document I showed you earlier?")
print(memory_result)


In [ ]:
# The memories are plain markdown files you can inspect
from pathlib import Path

memory_dir = Path(".strands/memory") / f"multimodal-{USER_ID}"
for f in sorted(memory_dir.glob("*.md")):
    print(f"--- {f.name}")
    print(f.read_text()[:300])


## Summary

In this notebook, you learned:

✅ How to give a Strands agent long-term memory with `MemoryManager` and `FileMemoryStore`

✅ How automatic extraction and the `add_memory` / `search_memory` tools work together

✅ How memories survive across sessions (new agent, same store)

✅ How to combine memory with image, video and document analysis

### Next Steps

Continue to the [S3 Vectors notebook](04-multimodal-with-s3-vectors.ipynb) for a cloud-backed memory implementation.
